# Data Preprocessing

In [2]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder \
    .appName("Final_Project_Data_Preprocessing") \
    .getOrCreate()

DATA_FILEPATH = '/standard/ds7200-apt4c/team2/ptbxl_nw'

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/apps/software/standard/core/pyspark/4.2.0-py3.14/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/06 17:09:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.parquet(DATA_FILEPATH)
df.select("ecg_id").show(5)

+------+
|ecg_id|
+------+
|     2|
|     4|
|     7|
|     8|
|    10|
+------+
only showing top 5 rows


### Data Types

In [4]:
from collections import Counter

dtype_counts = Counter(dtype for col, dtype in df.dtypes)

print(dtype_counts)

Counter({'double': 1863, 'int': 204, 'string': 17})


### Duplicated IDs: ECGs and Patients

In [28]:
import pyspark.sql.functions as fn

df.agg(
    fn.count('ecg_id').alias('count ecgs'),
    fn.countDistinct('ecg_id').alias('distinct ecgs')
).show()

+----------+-------------+
|count ecgs|distinct ecgs|
+----------+-------------+
|     21795|        21795|
+----------+-------------+



In [6]:
df.agg(
    fn.count('patient_id').alias('count patients'),
    fn.countDistinct('patient_id').alias('distinct patients')
).show()

+--------------+-----------------+
|count patients|distinct patients|
+--------------+-----------------+
|         21795|            18868|
+--------------+-----------------+



Can we keep the duplicate patients since there is a large enough sample size that they won't be influential?

### Missing Values

In [11]:
missing = df.agg(*[
    fn.sum(fn.col(c).isNull().cast('int')).alias(c)
    for c in df.columns
])

missing_count = (missing.select(
    fn.expr("stack({}, {})".format(
        len(df.columns),
        ", ".join([f"'{c}', `{c}`" for c in df.columns])))
    .alias("column", "missing_count"))
    .filter(fn.col("missing_count") > 100)
    .orderBy("missing_count", ascending=False)
)
missing_count.show(100)

26/11/06 17:21:30 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
26/11/06 17:21:43 WARN DAGScheduler: Broadcasting large task binary with size 4.4 MiB
[Stage 19:>                                                         (0 + 1) / 1]

+--------------------+-------------+
|              column|missing_count|
+--------------------+-------------+
| electrodes_problems|        21763|
| infarction_stadium2|        21688|
|unig_QRS_NotchAmp_V1|        21687|
| unig_QRS_NotchOn_V1|        21686|
|unig_QRS_NotchPea...|        21686|
|unig_QRS_NotchAmp_V2|        21673|
|unig_QRS_NotchAmp_V3|        21672|
| unig_QRS_NotchOn_V2|        21672|
| unig_QRS_NotchOn_V3|        21672|
|unig_QRS_NotchPea...|        21672|
|unig_QRS_NotchPea...|        21672|
|           pacemaker|        21508|
|unig_QRS_NotchAmp_V4|        21501|
| unig_QRS_NotchOn_V4|        21500|
|unig_QRS_NotchPea...|        21500|
|         burst_noise|        21179|
|unig_QRS_NotchAmp_V5|        20817|
| unig_QRS_NotchOn_V5|        20815|
|unig_QRS_NotchPea...|        20815|
|      baseline_drift|        20186|
|         extra_beats|        19847|
|unig_QRS_NotchAmp...|        19683|
|unig_QRS_NotchOn_III|        19682|
|unig_QRS_NotchPea...|        19682|
|

CONSULT DATA DICTIONARY

Could some of the columns have missing values where the patient does not have the specified characteristic? For example, does a missing pacemaker value mean they don't have a pacemaker?

If so, replace with 0 if the field is an integer or an indicator

Columns where this could apply: heart_axis, static_noise, baseline_drift, extra_beats, pacemaker, burst_noise, electrodes_problems, infarction_stadium2

#### Removing Columns with over 90% (19,616) Missing Values

In [10]:
0.9*21795

19615.5

In [23]:
high_missing = missing_count.filter(missing_count.missing_count >= 19615)
remove_cols = [row["column"] for row in high_missing.select("column").collect()]

# df = df.select([
#     c for c in df.columns if ~(c.isin(remove_cols))
# ])

26/11/06 17:29:42 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
26/11/06 17:29:49 WARN DAGScheduler: Broadcasting large task binary with size 4.4 MiB
                                                                                

['electrodes_problems',
 'infarction_stadium2',
 'unig_QRS_NotchAmp_V1',
 'unig_QRS_NotchOn_V1',
 'unig_QRS_NotchPeak_V1',
 'unig_QRS_NotchAmp_V2',
 'unig_QRS_NotchAmp_V3',
 'unig_QRS_NotchOn_V2',
 'unig_QRS_NotchOn_V3',
 'unig_QRS_NotchPeak_V2',
 'unig_QRS_NotchPeak_V3',
 'pacemaker',
 'unig_QRS_NotchAmp_V4',
 'unig_QRS_NotchOn_V4',
 'unig_QRS_NotchPeak_V4',
 'burst_noise',
 'unig_QRS_NotchAmp_V5',
 'unig_QRS_NotchOn_V5',
 'unig_QRS_NotchPeak_V5',
 'baseline_drift',
 'extra_beats',
 'unig_QRS_NotchAmp_III',
 'unig_QRS_NotchOn_III',
 'unig_QRS_NotchPeak_III']

#### Inserting zeros for missing values of indicators

In [ ]:
#df = df.fillna(0, subset=[""])

### Outliers?

In [25]:
df_outliers = df.select('ecg_id', 'age', 'height', 'weight').dropna()

cols = ['age', 'height', 'weight']
bounds = {}

for col in cols:
    quantiles = df_outliers.approxQuantile(
        col, [0.25, 0.75], 0.05
    )
    
    IQR = quantiles[1] - quantiles[0]
    
    bounds[col] = [
        quantiles[0] - 1.5 * IQR, 
        quantiles[1] + 1.5 * IQR
]

outliers = df_outliers.select(*['ecg_id'] + [
    (
        (df[c] < bounds[c][0]) | 
        (df[c] > bounds[c][1])
    ).alias(c + '_o') for c in cols
])
outliers.show()

+------+-----+--------+--------+
|ecg_id|age_o|height_o|weight_o|
+------+-----+--------+--------+
|   116|false|   false|   false|
|   136|false|   false|   false|
|   156|false|   false|   false|
|   256| true|   false|   false|
|   260|false|   false|   false|
|   261|false|   false|   false|
|   264|false|   false|   false|
|   265|false|   false|   false|
|   267|false|   false|   false|
|   270|false|   false|   false|
|   272|false|   false|   false|
|   273|false|   false|   false|
|   275|false|   false|   false|
|   276|false|   false|   false|
|   277|false|   false|   false|
|   278|false|   false|   false|
|   281|false|   false|   false|
|   284|false|   false|   false|
|   287|false|   false|   false|
|   288|false|   false|   false|
+------+-----+--------+--------+
only showing top 20 rows


In [26]:
df_outliers = df_outliers.join(outliers, on='ecg_id')
df_outliers.filter('weight_o').select('ecg_id', 'weight').show()
df_outliers.filter('age_o').select('ecg_id', 'age').show()

+------+------+
|ecg_id|weight|
+------+------+
|   751| 130.0|
|   856| 110.0|
|   911| 110.0|
|   975| 109.0|
|  1000| 120.0|
|  1083| 108.0|
|  1150| 118.0|
|  1185| 118.0|
|  1195| 112.0|
|  1213| 106.0|
|  1393| 110.0|
|  1625|  32.0|
|  1983| 119.0|
|  2081| 110.0|
|  2976| 140.0|
|  3526| 124.0|
|  3961|  25.0|
|  4634| 108.0|
|  4691| 109.0|
|  4759| 116.0|
+------+------+
only showing top 20 rows
+------+-----+
|ecg_id|  age|
+------+-----+
|   256|300.0|
|   351|300.0|
|   420| 22.0|
|   569|300.0|
|   637|300.0|
|   645| 19.0|
|   712|300.0|
|   730|300.0|
|   873|300.0|
|   932|300.0|
|  1075|300.0|
|  1184| 21.0|
|  1236| 20.0|
|  1502|300.0|
|  1625|300.0|
|  1657| 12.0|
|  1782| 22.0|
|  1792|300.0|
|  1970| 17.0|
|  1992|300.0|
+------+-----+
only showing top 20 rows


Age values of 300: can probably remove these since there are only a few hundred

In [27]:
df.filter('age >= 120').count()

293

In [ ]:
#df = df.filter('age <= 120')

### String Features

In [44]:
string_cols = ['report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'second_opinion',
               'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise',
               'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']

In [56]:
from pyspark.sql import functions as F

col = "infarction_stadium1"

df_valuecounts = (df
    .select(col)
    .groupby(col)
    .agg(F.count(col))
    .orderBy(col)
               )
df_valuecounts.show(100)

+--------------------+--------------------------+
| infarction_stadium1|count(infarction_stadium1)|
+--------------------+--------------------------+
|                NULL|                         0|
|                 RAO|                         1|
|          SEHYP 15)"|                         1|
|                 avf|                         1|
|                 LAD|                         1|
|           Stadium I|                       166|
|        Stadium I-II|                         5|
|          Stadium II|                        88|
|      Stadium II-III|                       943|
|         Stadium III|                       978|
|             unknown|                      3428|
|{'IMI': 80.0, 'ND...|                         1|
|     {'PACE': 100.0}|                         1|
+--------------------+--------------------------+



Notes on values:
 - heart_axis - 8 acronyms, 8k missing, about 20 records with weird values
 - infarction_stadium1 - 6 values, 16k missing, data leakage?
 - infarction_stadium2 - 3 values, 21k missing, data leakage?
 - second_opinion - to be removed, 21k False
 - initial_autogenerated_report - convert to binary, all but three records either true or false
 - validated_by_human - same as ^
 - baseline_drift - this has a ton of unique values, check data dictionary. Would have a ton of levels if categorical
 - static_noise - same as ^
 - burst_noise - same as ^
 - extra_beats - same as ^
 - electrodes_problems - to be removed, 21k missing, 
 - pacemaker - consult data dictionary, if missing = no pacemaker convert to binary
 - strat_fold - this is just the folds for cross-validation
 - filename_lr - drop, this is basically an id

#### Drop columns that will not be used for modeling (free text)

In [ ]:
#df = df.drop("report", "filename_lr", "filename_hr")